# Batched Kalman filters on GPU — Colab runner
Runtime → Change runtime type → **T4 GPU** (free tier is enough). Then run all cells.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
!nvcc --version | tail -2

## 1. Get the code and MathDx (cuBLASDx headers)

In [ ]:
!git clone https://github.com/medlemine2206/batched-kalman-gpu.git
%cd batched-kalman-gpu
!pip -q install nvidia-mathdx==25.6.0 matplotlib

## 2. Build (architecture detected automatically)

In [ ]:
import subprocess
cap = subprocess.check_output(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"]).decode().split()[0]
arch = cap.replace(".", "")
print("building for sm_" + arch)
!cmake -S . -B build -DKF_ARCH={arch} -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j 2>&1 | grep -E "error|registers|spill" | head -20

## 3. Correctness: CPU reference, layout logic, GPU vs CPU

In [ ]:
!cd build && ctest --output-on-failure

## 4. Benchmark (65 536 filters × 100 scans) and batch-size sweep

In [ ]:
!mkdir -p results
!./build/kf_bench --filters 65536 --steps 100 --csv results/bench.csv

In [ ]:
!./build/kf_bench --sweep --steps 100 --check 1024 --repeats 3 --csv results/sweep.csv

## 5. Plots

In [ ]:
!./build/test_cpu 2000 200 results/cpu_rmse.csv > /dev/null
!python scripts/plot_results.py --bench results/bench.csv --sweep results/sweep.csv --cpu results/cpu_rmse.csv --out results
from IPython.display import Image, display
for f in ["roofline.png", "sweep.png", "tracking.png"]:
    display(Image("results/" + f))

## 6. (Optional) Nsight Compute profile of one kernel
Shows DRAM throughput, occupancy, coalescing. May be unavailable on some Colab images.

In [ ]:
!ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section Occupancy \
     -k regex:kf_fused_kernel -c 2 ./build/kf_bench --only fused_aos --filters 65536 --steps 2 --repeats 1 | tail -60

In [ ]:
!ncu --section SpeedOfLight --section MemoryWorkloadAnalysis --section Occupancy \
     -k regex:kf_fused_kernel -c 2 ./build/kf_bench --only fused_soa --filters 65536 --steps 2 --repeats 1 | tail -60

Download `results/` and commit the CSVs and PNGs to the repo, then fill the results table of the README.

In [ ]:
!zip -qr results.zip results && echo ok
from google.colab import files; files.download("results.zip")